# LLM 8일차 (2) — 멀티 에이전트 RAG : 질문 재구성 → 문서 검색 → 답변 (상태의 `next` 로 길 안내)

수업 필기 `LangGraph_실습5` 를 정리 · 완성한 노트입니다. 필기는 **질문 재구성 체인까지** 만들고 멈춰 있어서, 교재 p.111~116 의 '핵심 변화 포인트 5가지' 대로 상태 · 노드 · 라우터 · 그래프 · 실행을 채웠습니다.
(교재 `AI Agent` **p.107~119** — 싱글 vs 멀티 에이전트 · 멀티 에이전트 핵심 변화 5가지 · 실습 : 멀티 에이전트 RAG 시스템)

```
START → rephrase_node ──next='retriever_node'──→ retriever_node ──next='answer_node'──→ answer_node ──next='end'──→ END
        (LLM : '그거' → 명확한 질문)            (함수 : ChromaDB 검색)              (LLM : 대화 기록 + 매뉴얼로 답)
         rephrased_query 채움                    context 채움                        messages 에 AI 답 추가
```

- 싱글 에이전트판은 [`langgraph_single_agent_rag_chatbot.ipynb`](./langgraph_single_agent_rag_chatbot.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩**(`jhgan/ko-sroberta-multitask`) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 검색 순위 · 거리는 수업의 `text-embedding-3-small` 과 다를 수 있고, 답변 문장은 gpt-5-nano 와 다르다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

싱글 에이전트 노트와 같다 — `pip install langgraph langchain-openai langchain-community langchain-chroma python-dotenv` (키 없이 돌릴 때 `langchain-huggingface sentence-transformers` 추가). 같은 폴더에 `faq_manual.csv` · `.env`. 키가 있으면 `# ⏳` 셀, 없으면 `# 🧪` 셀을 실행한다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph langchain-openai langchain-community langchain-chroma python-dotenv

## 1. 싱글 에이전트 vs 멀티 에이전트 (교재 p.107~110)

| | 싱글 에이전트 | 멀티 에이전트 |
|---|---|---|
| 구조 | LLM 하나가 도구 · 추론으로 처음부터 끝까지 | 역할이 정해진 여러 에이전트(노드)가 **상태를 공유**하며 협력 |
| 예 | 단일 RAG 고객 지원 챗봇 · 검색 API 요약 | 분류 / 기술 지원 / 환불 에이전트 · 기획 / 개발 / 테스트 에이전트 |
| 장점 | 단순 · 디버깅 쉬움 | 역할 분담 · 확장(새 에이전트 추가) 쉬움 |
| 단점 | 할 일이 많아지면 프롬프트 하나가 과부하 | 에이전트 간 통신 · 오류 전파 · 관리가 복잡 |

> **📝 교재 p.110 표 보충** '성능 : 멀티 — 병렬 처리로 향상 가능' 은 에이전트가 **동시에** 돌 때 이야기다. 오늘 실습처럼 재구성 → 검색 → 답변이 **차례로** 이어지면 LLM 호출이 1번(싱글) → 2번(재구성 + 답변)으로 늘어 **더 느리고 비싸다**. 얻는 것은 속도가 아니라 '그거' 같은 질문을 풀어내는 **정확도**다. 교재 p.114 '임무를 쪼개면 gpt-5-nano 급으로도 상용 성능' 도 이런 효과를 기대한다는 뜻이지 보장은 아니다 — 아래 ⏳ 셀로 직접 확인해 보자.
> 교재 p.108 그림(Single Agent · Network · Supervisor · Hierarchical · Custom)은 LangGraph 문서의 멀티 에이전트 구조 그림이다. 오늘 만드는 것은 정해진 순서로 넘겨주는 가장 단순한 형태다.

## 2. 환경 · 데이터 · 검색기 (필기와 같음)

In [2]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated, Literal
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END, add_messages

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
loaded = load_dotenv(find_dotenv(usecwd=True), override=True)
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


In [4]:
# CSV → Document (한 행 = 문서 1개) — 본문에는 질문 · 답, 메타데이터에는 번호 · 분류
loader = CSVLoader(
    file_path='faq_manual.csv',
    encoding='utf-8',
    content_columns=['question', 'answer'],          # page_content 에 넣을 열
    metadata_columns=['faq_id', 'category'],         # 📝 metadata 에 넣을 열 (필기에는 없음)
)
docs = loader.load()
print(f'로드된 문서 수 : {len(docs)}개')
print(docs[1].page_content)
print(docs[1].metadata)

로드된 문서 수 : 10개
question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
answer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.
{'source': 'faq_manual.csv', 'row': 1, 'faq_id': 'FAQ-002', 'category': '결제/환불'}


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import OpenAIEmbeddings
model = OpenAIEmbeddings(model='text-embedding-3-small')    # 1536차원 · 입력 최대 8,191 토큰

In [6]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                              encode_kwargs={'normalize_embeddings': True})
print('벡터 길이 :', len(model.embed_query('테스트')))

벡터 길이 : 768


In [7]:
vectorstore = Chroma.from_documents(documents=docs, embedding=model, collection_name='faq_multi')   # 메모리 저장소 (폴더 없음)
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})
print('문서 수 :', vectorstore._collection.count(), '| 환불은 →', retriever.invoke('환불은')[0].metadata['faq_id'])

문서 수 : 10 | 환불은 → FAQ-002


## 3. 에이전트별 전용 체인 (교재 p.114 · p.116 · 필기)

- `rephrase_chain` — 대화 기록을 보고 **질문만 고친다** (답하지 말 것)
- `qa_chain` — 찾은 매뉴얼로 **고객 응대만** 한다
- 둘 다 `MessagesPlaceholder('chat_history')` 로 **과거 대화가 들어갈 자리**를 만든다 (교재 p.116 '전달 경로')

In [8]:
rephrase_system_prompt = '''
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?")을,
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: "수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?")으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
'''
rephrase_prompt = ChatPromptTemplate.from_messages([
    ('system', rephrase_system_prompt),
    MessagesPlaceholder(variable_name='chat_history'),     # 과거 대화 [Human, AI, Human, AI, …] 가 들어갈 자리
    ('human', '{input}'),
])

qa_system_prompt = '''
당신은 친절한 온라인 교육 플랫폼 직원입니다.
제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.
[컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.
'''
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', qa_system_prompt),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '[컨텍스트]\n{context}\n\n질문:\n{question}'),
])
print('rephrase 입력 :', rephrase_prompt.input_variables, '| qa 입력 :', qa_prompt.input_variables)

# 대화 기록이 프롬프트의 어디에 들어가는지
demo = rephrase_prompt.invoke({'chat_history': [HumanMessage('환불은 어떻게?'), AIMessage('7일 이내 …')],
                               'input': '그거 전액 다 받으려면 조건이 뭐야?'})
for m in demo.to_messages():
    print(f'  {type(m).__name__:<14}{m.content.strip()[:40]}')

rephrase 입력 : ['chat_history', 'input'] | qa 입력 : ['chat_history', 'context', 'question']
  SystemMessage 주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때
  HumanMessage  환불은 어떻게?
  AIMessage     7일 이내 …
  HumanMessage  그거 전액 다 받으려면 조건이 뭐야?


> **📝 고친 부분** 필기 시스템 프롬프트 '모호한 질문(예: …), 채팅 기록 없이도 이해할 수 있는 …으로' 는 '모호한 질문**을**' 의 조사가 빠져 문장이 끊겨 있었다. 필기 `print(rephrase_prompt)` 는 긴 타입 정보만 찍혀 읽기 어려워, 실제로 만들어지는 메시지 순서를 찍도록 바꿨다(System → 과거 Human · AI → 지금 질문).

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-5-nano')         # 두 체인이 같은 LLM 을 쓴다 (필기)

In [10]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용

# 재구성 : 앞 사용자 질문을 지금 질문 앞에 '이어 붙이는' 단순 규칙 (진짜 LLM 은 문장을 새로 쓴다)
# 답변   : 컨텍스트의 답 첫 문장 + 받은 대화 기록 수
from langchain_core.runnables import RunnableLambda

def _fake_llm(prompt_value):
    msgs = prompt_value.to_messages()
    system, last = msgs[0].content, msgs[-1].content
    history = msgs[1:-1]
    if '재구성' in system:
        prev = [m.content for m in history if isinstance(m, HumanMessage)]
        return AIMessage(f'{prev[-1]} {last}' if prev else last)
    context = last.split('[컨텍스트]')[-1].split('질문:')[0]
    first = context.split('answer: ')[-1].split('. ')[0].strip() if 'answer: ' in context else '(없음)'
    return AIMessage(f'(모의 답변 · 대화 기록 {len(history)}개 참고) {first}.')

llm = RunnableLambda(_fake_llm)

In [11]:
rephrase_chain = rephrase_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()

## 4. 상태 · 노드 · 라우터 · 그래프 (교재 p.111~115 · 필기가 멈춘 곳부터)

### 변화 2 — 상태 확장 (p.112) · 변화 3 — `next` 로 행선지 기록 (p.113)

In [12]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []   # 대화 기록 (누적)
    rephrased_query: str = ''      # rephrase_node → retriever_node 로 넘기는 정제된 질문
    context: str = ''              # retriever_node → answer_node 로 넘기는 매뉴얼 본문
    next: str = ''                 # 각 노드가 적어 두는 '다음 노드 이름' — 라우터가 읽는다

### 변화 1 — 노드 3개로 나누기 (p.111) · 변화 5 — 과거 대화 전달 (p.115~116)

'판단 · 추론' 이 필요한 재구성 · 답변은 **LLM 노드**, 정해진 일을 하는 검색은 **함수 노드**(p.111 ⑥).

In [13]:
def split_history(state: AgentState):
    # 전체 메시지 = 과거 대화 + 마지막 질문 → 둘로 나눠 체인에 넘긴다 (교재 p.115 ⑤)
    return state.messages[:-1], state.messages[-1].content

def rephrase_node(state: AgentState) -> dict:
    chat_history, question = split_history(state)
    rephrased = rephrase_chain.invoke({'chat_history': chat_history, 'input': question})
    print(f'  [1 rephrase ] {question!r} → {rephrased!r}')
    return {'rephrased_query': rephrased, 'next': 'retriever_node'}

def retriever_node(state: AgentState) -> dict:
    docs_found = retriever.invoke(state.rephrased_query)          # 원래 질문이 아니라 '고친 질문' 으로 검색
    context = docs_found[0].page_content if docs_found else ''
    print(f"  [2 retriever] → {docs_found[0].metadata['faq_id'] if docs_found else '없음'} | {context.splitlines()[0][10:45] if context else ''}")
    return {'context': context, 'next': 'answer_node'}

def answer_node(state: AgentState) -> dict:
    chat_history, question = split_history(state)
    answer = qa_chain.invoke({'chat_history': chat_history, 'context': state.context, 'question': question})
    print(f'  [3 answer   ] {answer[:70]}')
    return {'messages': [AIMessage(content=answer)], 'next': 'end'}

def router(state: AgentState) -> str:
    return state.next                    # 노드가 적어 둔 행선지를 그대로 읽는다

In [14]:
workflow = StateGraph(AgentState)
workflow.add_node('rephrase_node', rephrase_node)
workflow.add_node('retriever_node', retriever_node)
workflow.add_node('answer_node', answer_node)
workflow.set_entry_point('rephrase_node')

workflow.add_conditional_edges('rephrase_node', router, {'retriever_node': 'retriever_node'})
workflow.add_conditional_edges('retriever_node', router, {'answer_node': 'answer_node'})
workflow.add_conditional_edges('answer_node', router, {'end': END})

agent = workflow.compile()
print(agent.get_graph().draw_mermaid().split('graph TD;')[1].split('classDef')[0])


	__start__([<p>__start__</p>]):::first
	rephrase_node(rephrase_node)
	retriever_node(retriever_node)
	answer_node(answer_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> rephrase_node;
	answer_node -. &nbsp;end&nbsp; .-> __end__;
	rephrase_node -.-> retriever_node;
	retriever_node -.-> answer_node;
	


- 매핑 dict 덕분에 그림에 점선(조건부)으로 정확히 그려지고, 노드가 `next` 에 엉뚱한 이름을 적으면 `KeyError` 로 바로 드러난다(Day 052)
- 📝 지금은 갈 곳이 노드마다 **하나뿐**이라 `add_edge` 세 줄과 결과가 같다. `next` 방식이 빛나는 때는 노드가 **실행 중에 행선지를 고를 때**다 — 예 : 재구성 노드가 '인사' 라고 판단하면 `next='answer_node'` 로 검색을 건너뛰기, 검색 결과가 없으면 `next='rephrase_node'` 로 되돌아가기(이때는 Day 052 의 `recursion_limit` 안전장치를 함께)

## 5. 실행 — 대화를 이어서 두 번 묻기 (교재 p.117~119 실습)

LangGraph 는 `invoke` 사이에 상태를 기억하지 않는다(체크포인터 없음). 그래서 **앞 결과의 `messages` 에 새 질문을 붙여** 다음 `invoke` 에 넣는다.

In [15]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

chat = []                                                       # 누적 대화 기록
for question in ['환불은 어떻게?', '그거 전액 다 받으려면 조건이 뭐야?']:
    print(f'질문 : {question}')
    result = agent.invoke({'messages': chat + [HumanMessage(content=question)]})
    chat = result['messages']                                   # [Human, AI, Human, AI, …]
    print(f"  → messages {len(chat)}개 | rephrased_query = {result['rephrased_query']!r}\n")

질문 : 환불은 어떻게?
  [1 rephrase ] '환불은 어떻게?' → '환불은 어떻게?'
  [2 retriever] → FAQ-002 | 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요
  [3 answer   ] (모의 답변 · 대화 기록 0개 참고) 네, 환불 가능합니다.
  → messages 2개 | rephrased_query = '환불은 어떻게?'

질문 : 그거 전액 다 받으려면 조건이 뭐야?
  [1 rephrase ] '그거 전액 다 받으려면 조건이 뭐야?' → '환불은 어떻게? 그거 전액 다 받으려면 조건이 뭐야?'
  [2 retriever] → FAQ-002 | 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요
  [3 answer   ] (모의 답변 · 대화 기록 2개 참고) 네, 환불 가능합니다.
  → messages 4개 | rephrased_query = '환불은 어떻게? 그거 전액 다 받으려면 조건이 뭐야?'



- 두 번째 질문은 재구성 덕분에 **환불 FAQ(FAQ-002)** 를 찾았다. 싱글 에이전트 노트에서 같은 질문은 수료증(FAQ-003)이었다
- 🧪 가짜 재구성은 '앞 질문 + 지금 질문' 을 이어 붙였을 뿐이다. 아래처럼 이 규칙은 **앞 주제가 바뀌면** 틀린다 — 진짜 LLM 재구성이 필요한 이유

In [16]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

chat = []
for question in ['강의 영상이 안 나와요', '그럼 그건 앱에서도 돼?']:
    print(f'질문 : {question}')
    result = agent.invoke({'messages': chat + [HumanMessage(content=question)]})
    chat = result['messages']
print()
for q in ['그럼 그건 앱에서도 돼?', '강의 영상이 안 나와요 그럼 그건 앱에서도 돼?', '모바일 앱으로도 강의를 들을 수 있나요?']:
    (doc, dist), = vectorstore.similarity_search_with_score(q, k=1)
    print(f'{q:<32} → {doc.metadata["faq_id"]} 거리 {dist:.3f}')

질문 : 강의 영상이 안 나와요
  [1 rephrase ] '강의 영상이 안 나와요' → '강의 영상이 안 나와요'


  [2 retriever] → FAQ-005 | 강의 동영상이 자꾸 끊기거나 재생되지 않아요.
  [3 answer   ] (모의 답변 · 대화 기록 0개 참고) 동영상 재생에 불편을 드려 죄송합니다.
질문 : 그럼 그건 앱에서도 돼?
  [1 rephrase ] '그럼 그건 앱에서도 돼?' → '강의 영상이 안 나와요 그럼 그건 앱에서도 돼?'


  [2 retriever] → FAQ-005 | 강의 동영상이 자꾸 끊기거나 재생되지 않아요.
  [3 answer   ] (모의 답변 · 대화 기록 2개 참고) 동영상 재생에 불편을 드려 죄송합니다.

그럼 그건 앱에서도 돼?                    → FAQ-010 거리 0.731


강의 영상이 안 나와요 그럼 그건 앱에서도 돼?       → FAQ-005 거리 0.827


모바일 앱으로도 강의를 들을 수 있나요?           → FAQ-010 거리 0.296


- '앱에서도 돼?' 만으로는 모바일 앱 FAQ(FAQ-010)를 찾지만, 앞 질문을 이어 붙이면 앞 주제(동영상 재생 FAQ-005)가 이겨 버린다. LLM 이 '모바일 앱으로도 강의를 들을 수 있나요?' 처럼 **한 문장으로 다시 써야** 맞는 FAQ 를 찾는다

### 노드마다 상태가 어떻게 채워지나 — `stream(stream_mode='updates')`

In [17]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용

history = [HumanMessage('환불은 어떻게?'), AIMessage('결제일로부터 7일 이내 …')]
for update in agent.stream({'messages': history + [HumanMessage('그거 전액 다 받으려면 조건이 뭐야?')]}, stream_mode='updates'):
    node, changed = next(iter(update.items()))
    shown = {k: (v[-1].content[:30] if k == 'messages' else str(v)[:40]) for k, v in changed.items()}
    print(f'{node:<15}{shown}')

  [1 rephrase ] '그거 전액 다 받으려면 조건이 뭐야?' → '환불은 어떻게? 그거 전액 다 받으려면 조건이 뭐야?'
rephrase_node  {'rephrased_query': '환불은 어떻게? 그거 전액 다 받으려면 조건이 뭐야?', 'next': 'retriever_node'}


  [2 retriever] → FAQ-002 | 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요
retriever_node {'context': 'question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻', 'next': 'answer_node'}
  [3 answer   ] (모의 답변 · 대화 기록 2개 참고) 네, 환불 가능합니다.
answer_node    {'messages': '(모의 답변 · 대화 기록 2개 참고) 네, 환불 가능', 'next': 'end'}


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 진짜 gpt-5-nano 로 같은 대화 — 재구성 결과(rephrased_query)와 찾은 FAQ 를 확인
llm = ChatOpenAI(model='gpt-5-nano')
rephrase_chain = rephrase_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()
chat = []
for question in ['환불은 어떻게?', '그거 전액 다 받으려면 조건이 뭐야?', '그럼 그건 앱에서도 돼?']:
    result = agent.invoke({'messages': chat + [HumanMessage(content=question)]})
    chat = result['messages']
    print(f"질문 : {question}\n재구성 : {result['rephrased_query']}\n답 : {chat[-1].content}\n")

## 정리 — 멀티 에이전트 적용 시 핵심 변화 5가지 (교재 p.111~116)

| 변화 | 싱글 (`LangGraph_실습4`) | 멀티 (이 노트) |
|---|---|---|
| 1 노드 | `rag_node` 하나 | `rephrase_node`(LLM) · `retriever_node`(함수) · `answer_node`(LLM) |
| 2 상태 | `messages` | + `rephrased_query` · `context` · `next` |
| 3 흐름 | `START → rag_node → END` | 노드가 `{'next': '다음 노드'}` 를 적고 `router` 가 읽어 조건부 엣지로 연결 |
| 4 체인 | `rag_chain` 하나 | `rephrase_chain`(질문 교정만) · `qa_chain`(응대만) |
| 5 대화 기록 | 프롬프트에 `context` · `question` 만 | `MessagesPlaceholder('chat_history')` + 노드에서 `messages[:-1]` / `messages[-1]` 로 나눔 · `invoke` 때 앞 `messages` 를 이어 넣음 |